# Fraud Risk — LoRA Fine-tuning (Colab)

Fine-tunes `google/gemma-2b` with LoRA on the 200-row balanced fundraiser dataset  
for binary fraud classification. Replaces the GCP Compute Engine GPU VM workflow.

**Runtime:** Connect to a T4 GPU runtime (`Runtime → Change runtime type → T4 GPU`).

**HuggingFace prerequisite:** Accept the Gemma license at https://huggingface.co/google/gemma-2b,  
then set your token in the cell below.

**Output:** LoRA adapter saved to `gs://fraud-risk-models/lora-adapters/v1/final-adapter`

In [1]:
# ── 1. Install deps & authenticate ────────────────────────────────────────────
!pip install -q peft transformers accelerate datasets mlflow google-cloud-storage

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GCP auth (opens a browser prompt — click the link, paste the code)
from google.colab import auth
auth.authenticate_user()

# HuggingFace token — required for gated Gemma model
# Get yours at https://huggingface.co/settings/tokens
from huggingface_hub import login
from google.colab import userdata
HF_TOKEN = userdata.get('gemma_hf')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
else:
    raise ValueError("Set HF_TOKEN above — Gemma requires a HuggingFace token.")

print("Setup complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [2]:
# ── 2. Config (mirrors ml_training_service/configs/lora_config.yaml) ──────────
GCP_PROJECT       = "project-8f2c61ff-718f-43ed-826"   # your GCP project ID, e.g. "my-fraud-project"
GCS_DATA_PATH     = "gs://fraud-risk-data/technique2_train_balanced_200.csv"
GCS_ADAPTER_OUT   = "gs://fraud-risk-models/lora-adapters/v1"
OUTPUT_DIR        = "/content/lora-adapter"

BASE_MODEL        = "google/gemma-2b"
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.1
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 10
BATCH_SIZE        = 1    # reduced from 8 — T4 OOM with Gemma-2B at bs=8
GRAD_ACCUM_STEPS  = 8    # effective batch size = 1 * 8 = 8
MAX_LENGTH        = 256  # reduced from 512 — halves activation memory

In [3]:
# ── 3. Load training data ─────────────────────────────────────────────────────
import pandas as pd

def load_data():
    """Try GCS first; fall back to manual Colab file upload."""
    try:
        df = pd.read_csv(GCS_DATA_PATH, storage_options={"token": "google_default"})
        print(f"Loaded {len(df)} rows from GCS: {GCS_DATA_PATH}")
        return df
    except Exception as e:
        print(f"GCS unavailable ({e})")
        print("Upload technique2_train_balanced_200.csv from your local machine:")
        from google.colab import files
        uploaded = files.upload()
        df = pd.read_csv(next(iter(uploaded)))
        print(f"Loaded {len(df)} rows from upload")
        return df

df = load_data()
print(f"Columns: {list(df.columns)}")
print(f"Label distribution: {df['label'].value_counts().to_dict()}")

GCS unavailable (b/fraud-risk-data/o)
Upload technique2_train_balanced_200.csv from your local machine:


Saving technique2_train_balanced_200.csv to technique2_train_balanced_200.csv
Loaded 200 rows from upload
Columns: ['fund_id', 'category_id', 'goal', 'descr_len', 'title_len', 'primary_phone_checks__line_type', 'identity_check_score', 'primary_email_address_checks__is_disposable', 'primary_email_address_checks__email_domain_creation_days', 'label', 'title', 'description', 'description_clean']
Label distribution: {0: 100, 1: 100}


In [4]:
# ── 4. Prepare records and train/eval split ───────────────────────────────────
from sklearn.model_selection import train_test_split

def build_records(df: pd.DataFrame) -> list[dict]:
    records = []
    for _, row in df.iterrows():
        title = str(row.get("title", "") or "")
        # use description_clean (HTML-stripped); fall back to raw description
        desc = str(row.get("description_clean", "") or row.get("description", "") or "")
        records.append({
            "fund_id":    str(row.get("fund_id", "")),
            "input_text": f"Title: {title}\n\nDescription: {desc}",
            "label":      int(row["label"]),
        })
    return records

all_records = build_records(df)
labels = [r["label"] for r in all_records]

train_records, eval_records = train_test_split(
    all_records, test_size=0.2, random_state=42, stratify=labels
)
print(f"Train: {len(train_records)}  Eval: {len(eval_records)}")
print(f"Sample input (truncated): {train_records[0]['input_text'][:200]}")

Train: 160  Eval: 40
Sample input (truncated): Title: Help with Lance’s funeral

Description: Hello everyone, This past Tuesday Lance Love Henry took his own life.My sister,mother and I need help to pay for the funeral costs so that he could be se


In [5]:
# ── 5. LoRA fine-tuning ───────────────────────────────────────────────────────
import os
import torch
import mlflow
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import roc_auc_score, f1_score

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

assert torch.cuda.is_available(), "No GPU detected — connect to a T4 runtime first."
print(f"GPU: {torch.cuda.get_device_name(0)}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "auc_roc": roc_auc_score(labels, probs),
        "f1":      f1_score(labels, preds, zero_division=0),
    }


def tokenize_dataset(records, tokenizer, max_length):
    ds = Dataset.from_list(records)
    def _tokenize(batch):
        return tokenizer(
            batch["input_text"],
            truncation=True, padding="max_length", max_length=max_length,
        )
    return ds.map(_tokenize, batched=True, remove_columns=["input_text", "fund_id"])

GPU: Tesla T4


In [8]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [9]:
# Build model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

GemmaForSequenceClassification LOAD REPORT from: google/gemma-2b
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,847,296 || all params: 2,508,023,808 || trainable%: 0.0737


In [10]:
# Tokenize
train_ds = tokenize_dataset(train_records, tokenizer, MAX_LENGTH)
eval_ds  = tokenize_dataset(eval_records,  tokenizer, MAX_LENGTH)

Map:   0%|          | 0/160 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [11]:
# SQLite tracking — file store is deprecated in mlflow>=3
mlflow.set_tracking_uri(f"sqlite:///{OUTPUT_DIR}/mlflow.db")
mlflow.set_experiment("fraud-lora-colab")

2026/06/04 19:09:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/04 19:09:22 INFO mlflow.store.db.utils: Updating database tables
2026/06/04 19:09:28 INFO mlflow.tracking.fluent: Experiment with name 'fraud-lora-colab' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1780600168037, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780600168037, lifecycle_stage='active', name='fraud-lora-colab', tags={}, trace_location=None, workspace='default'>

In [14]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="auc_roc",
    greater_is_better=True,
    fp16=False,   # Turn this off
    bf16=True,
    logging_steps=10,
    report_to="none",
)

In [15]:
with mlflow.start_run():
    mlflow.log_params({
        "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT, "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "train_size": len(train_records), "eval_size": len(eval_records),
    })

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
    )
    print("Params all set. Begin training..")
    trainer.train()
    print("Training done..")

    eval_results = trainer.evaluate()
    print("\nFinal eval:", eval_results)
    mlflow.log_metrics({k.replace("eval_", ""): v for k, v in eval_results.items()})

    adapter_path = f"{OUTPUT_DIR}/final-adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    mlflow.log_artifact(adapter_path, artifact_path="lora-adapter")

print(f"\nAdapter saved to: {adapter_path}")

Params all set. Begin training..


Epoch,Training Loss,Validation Loss,Auc Roc,F1
1,7.577798,1.098289,0.417500,0.342857
2,4.775500,0.821817,0.632500,0.585366
3,2.978545,0.692887,0.750000,0.708333
4,2.130297,0.707052,0.805000,0.680851
5,0.697896,0.653561,0.805000,0.714286
6,0.197395,0.898256,0.817500,0.777778
7,0.009899,1.051534,0.832500,0.736842
8,0.176214,1.069007,0.852500,0.780488


Epoch,Training Loss,Validation Loss,Auc Roc,F1
1,7.577798,1.098289,0.417500,0.342857
2,4.775500,0.821817,0.632500,0.585366
3,2.978545,0.692887,0.750000,0.708333
4,2.130297,0.707052,0.805000,0.680851
5,0.697896,0.653561,0.805000,0.714286
6,0.197395,0.898256,0.817500,0.777778
7,0.009899,1.051534,0.832500,0.736842
8,0.176214,1.069007,0.852500,0.780488
9,0.074805,1.173527,0.847500,0.761905
10,0.056680,1.185219,0.850000,0.761905


Training done..



Final eval: {'eval_loss': 1.0690066814422607, 'eval_auc_roc': 0.8525, 'eval_f1': 0.7804878048780488, 'eval_runtime': 2.707, 'eval_samples_per_second': 14.776, 'eval_steps_per_second': 14.776, 'epoch': 10.0}

Adapter saved to: /content/lora-adapter/final-adapter


In [17]:
# ── 6. Push adapter to GCS ────────────────────────────────────────────────────
import subprocess

print(f"Uploading adapter to {GCS_ADAPTER_OUT} ...")
result = subprocess.run(
    ["gsutil", "-m", "cp", "-r", f"{OUTPUT_DIR}/final-adapter", GCS_ADAPTER_OUT],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print("Done!")
    print(f"\nTo download locally and run evaluation:")
    print(f"  gsutil -m cp -r {GCS_ADAPTER_OUT}/final-adapter models/lora-adapter/")
    print(f"  LORA_ADAPTER_PATH=models/lora-adapter/final-adapter make serve")

Uploading adapter to gs://fraud-risk-models/lora-adapters/v1 ...
Done!

To download locally and run evaluation:
  gsutil -m cp -r gs://fraud-risk-models/lora-adapters/v1/final-adapter models/lora-adapter/
  LORA_ADAPTER_PATH=models/lora-adapter/final-adapter make serve
